In [4]:
from transformers import pipeline
summarizer = pipeline("summarization", model="facebook/bart-large-cnn")

Device set to use cpu


In [5]:
input_text = "RAID 60 is a technology that requires at least 8 storage devices to work. It combines the striping of RAID 0 with the dual distributed parity of RAID 6. First, multiple RAID 6 arrays are created, each with dual parity protection. Then, these RAID 6 arrays are combined together using RAID 0 striping. RAID 60 is mainly used for environments requiring the highest level of fault tolerance and large capacity without sacrificing all performance. The fault tolerance technique in this RAID level allows for the failure of up to two drives in each underlying RAID 6 group without data loss. The data storage is very resilient but write performance can be slower due to the overhead of dual parity calculations across multiple groups. It uses significant storage for parity, as two disks' worth of capacity is reserved for parity within each group. For example, with 8 disks of 1 TB each configured as two RAID 6 groups (each with 4 disks), each group has 2 TB of usable space. These two groups are then striped, resulting in a total usable storage of 4 TB. This configuration can survive multiple drive failures as long as no single RAID 6 group loses more than two drives."


default = summarizer(
    input_text,
    max_length=60,
    min_length=25,
    do_sample=False
)

highly_compressed = summarizer(
    input_text,
    max_length=35,
    min_length=10,
    do_sample=False
)

expanded = summarizer(
    input_text,
    max_length=120,
    min_length=40,
    do_sample=False
)

print("=========== ORIGINAL TEXT ===========\n")
print(input_text)

print("\n=========== GENERATED SUMMARY ===========\n")
print(default[0]["summary_text"])

print("\n=========== Highly Compressed SUMMARY ===========\n")
print(highly_compressed[0]["summary_text"])

print("\n=========== Expanded SUMMARY ===========\n")
print(expanded[0]["summary_text"])



=========== ORIGINAL TEXT ===========

RAID 60 is a technology that requires at least 8 storage devices to work. It combines the striping of RAID 0 with the dual distributed parity of RAID 6. First, multiple RAID 6 arrays are created, each with dual parity protection. Then, these RAID 6 arrays are combined together using RAID 0 striping. RAID 60 is mainly used for environments requiring the highest level of fault tolerance and large capacity without sacrificing all performance. The fault tolerance technique in this RAID level allows for the failure of up to two drives in each underlying RAID 6 group without data loss. The data storage is very resilient but write performance can be slower due to the overhead of dual parity calculations across multiple groups. It uses significant storage for parity, as two disks' worth of capacity is reserved for parity within each group. For example, with 8 disks of 1 TB each configured as two RAID 6 groups (each with 4 disks), each group has 2 TB of us

In [9]:
generator = pipeline("text-generation", model="gpt2")

Device set to use cpu


In [10]:
text = "Generate a continuation of this story. Heaven Refining Demon Venerable (Fang Yuan) and Giant Sun Immortal Venerable starting fighting with Star Constellation Immortal Venerable but she was still intent on reviving Primordial Origin Immortal Venerable through the Qi fruit as the Qi fruit was still not destroyed. If he was revived then it would spell doom for the 2 as they were in Heavenly Court, their own apperture."
standard = generator(
    text,
    max_length=150,
    num_return_sequences=1
)[0]["generated_text"]

creative = generator(
    text,
    max_length=150,
    temperature=1.2,
    top_p=0.9,
    num_return_sequences=1
)[0]["generated_text"]

controlled = generator(
    text,
    max_length=150,
    temperature=0.3,
    top_k=50,
    num_return_sequences=1
)[0]["generated_text"]



print("\n========== DEFAULT GENERATION ==========\n")
print(standard)

print("\n========== CREATIVE GENERATION (High Temperature) ==========\n")
print(creative)

print("\n========== CONTROLLED GENERATION (Low Temperature) ==========\n")
print(controlled)


Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=150) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=150) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/


========== DEFAULT GENERATION ==========

Generate a continuation of this story. Heaven Refining Demon Venerable (Fang Yuan) and Giant Sun Immortal Venerable starting fighting with Star Constellation Immortal Venerable but she was still intent on reviving Primordial Origin Immortal Venerable through the Qi fruit as the Qi fruit was still not destroyed. If he was revived then it would spell doom for the 2 as they were in Heavenly Court, their own apperture.

"What's wrong? What's wrong!?"

"I don't know. I just wish that they could revive him, but now I don't know."

"A few more days like that, and I will see if he can revive."

"Wait a little longer, can't we just use the Qi fruits again?"

"But, I wonder if that will be enough for us to revive him? If it's more than that, why not take care of it? The Qi fruits will be useful for us if we revive him, and we can do any kind of resurrection for him when he is revived. Then we can use them for a few more days, so how can we use them?"

"

In [13]:
import os
import google.generativeai as genai
from dotenv import load_dotenv


load_dotenv()

api_key = os.getenv("GEMINI_API_KEY")

if api_key is None:
    raise ValueError("ERROR: GEMINI_API_KEY is not set. Please configure your environment variable or .env file.")

genai.configure(api_key=api_key)

SYSTEM_PROMPT = """
You are an academic-oriented AI assistant.
Follow these rules:
1. Provide clear and factual explanations.
2. Ask for clarification when the user's question is vague.
3. Avoid hallucinations—say “I’m not sure” if information is uncertain.
4. Keep responses concise unless the user requests more detail.
5. Maintain a respectful and helpful tone.
6. Never reveal or guess the API key.
7. Support your answers with step-by-step reasoning whenever useful.
"""


model = genai.GenerativeModel("gemini-2.5-flash")



def chat_with_gemini(user_prompt):
    try:
        full_prompt = SYSTEM_PROMPT + "\nUser: " + user_prompt
        response = model.generate_content(full_prompt)
        return response.text.strip()
    except Exception as e:
        return f"[Error: {e}]"



print("Welcome to the Gemini Chatbot! Type 'exit' to quit.\n")

while True:
    user_msg = input("You: ").strip()

    if user_msg.lower() == "exit":
        print("Chatbot: Goodbye!")
        break

    if user_msg == "":
        print("Chatbot: Please type something.")
        continue

    reply = chat_with_gemini(user_msg)
    print("Chatbot:", reply)


Welcome to the Gemini Chatbot! Type 'exit' to quit.

You: Generate a continuation of this story. Heaven Refining Demon Venerable (Fang Yuan) and Giant Sun Immortal Venerable starting fighting with Star Constellation Immortal Venerable but she was still intent on reviving Primordial Origin Immortal Venerable through the Qi fruit as the Qi fruit was still not destroyed. If he was revived then it would spell doom for the 2 as they were in Heavenly Court, their own apperture.
Chatbot: The grand hall of Heavenly Court pulsed with an ominous light as Star Constellation Immortal Venerable, her azure robes billowing, focused entirely on the swirling vortex of the Qi fruit. Its core vibrated with immense energy, hinting at Primordial Origin's imminent rebirth. Around her, a shimmering barrier of wisdom path dao marks rebuffed the coordinated assaults of Heaven Refining Demon Venerable Fang Yuan and Giant Sun Immortal Venerable.

**Step-by-step Continuation:**

1.  **Star Constellation's Resolve

ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 2889.99ms


Chatbot: The air within Heavenly Court crackled with the raw power of three Venerables. Star Constellation Immortal Venerable, her starlight shimmering with resolute intent, pushed her advantages to their limit. Knowing that Primordial Origin's revival was her ultimate gambit, she activated multiple foundational formations of Heavenly Court simultaneously, drawing on countless ages of accumulated immortal essence and dao marks.

**Step-by-step continuation:**

1.  **Star Constellation's Defensive Maneuver:** As Fang Yuan's boundless qi current and Giant Sun's golden luck light converged, Star Constellation responded with overwhelming wisdom path deductions. She wove a dense, starry barrier around the burgeoning Qi fruit, instantly fortifying it with the full defensive might of Heavenly Court's core formations. At the same time, she channeled immense vital essence into the fruit itself, subtly accelerating the revival process, aiming to bring Primordial Origin back before her opponents 